# How much of a protein can you change before it stops being that protein?

**Protein design practical — University of Tartu**

You will take a real protein, delete part of its sequence, and ask a generative
model to invent replacements. Then you will measure what survived: how much of
the original sequence came back, whether the amino-acid composition still looks
like a protein, and whether the fold held.

Along the way you will catch the model doing something it should not.

---

## How to use this notebook

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Fill in your choices in the **one cell below**.
3. **Runtime → Run all**, then leave it. Setup downloads ~5 GB of weights and
   takes several minutes; everything after that is quick.

Everything after the choices cell runs by itself. If no GPU is available the
notebook falls back to CPU automatically — it still works, just more slowly, so
stick to the shorter proteins.


In [ ]:
#@title ## YOUR CHOICES — set these, then Runtime → Run all { display-mode: "form" }

#@markdown ### 1 · Which protein do you want to attack?
protein = "Ubiquitin (76 aa) - the classic signalling protein" #@param ["Trp-cage (20 aa) - the smallest thing that still folds", "Villin headpiece (36 aa) - a three-helix bundle", "Protein G B1 domain (56 aa) - half sheet, half helix", "Ubiquitin (76 aa) - the classic signalling protein", "GFP (238 aa) - a beta-barrel, slow on CPU", "paste my own below"]
my_own_sequence = "" #@param {type:"string"}
#@markdown Only needed if you chose *paste my own below*. One-letter code, no spaces.

#@markdown ### 2 · How much of it do you delete?
mask_fraction = 0.3 #@param {type:"slider", min:0.05, max:0.95, step:0.05}
#@markdown That fraction of residues is blanked out and ESM3 must fill them back
#@markdown in. Start around 30%, then come back and push it.

#@markdown ### 3 · How adventurous should the model be?
temperature = 0.7 #@param ["0.3", "0.7", "1.0", "1.5"] {type:"raw"}
#@markdown Low = plays it safe and repeats itself. High = invents more, and
#@markdown eventually invents nonsense.
seed = 0 #@param {type:"integer"}
#@markdown Change the seed to mask a *different* random set of residues.

#@markdown ### 4 · Also sweep the masking fraction?
run_sweep = True #@param {type:"boolean"}
sweep_protein = "Ubiquitin (76 aa) - the classic signalling protein" #@param ["Trp-cage (20 aa) - the smallest thing that still folds", "Villin headpiece (36 aa) - a three-helix bundle", "Protein G B1 domain (56 aa) - half sheet, half helix", "Ubiquitin (76 aa) - the classic signalling protein", "GFP (238 aa) - a beta-barrel, slow on CPU"]
#@markdown One run tells you little. The sweep repeats the experiment across many
#@markdown masking fractions and plots where the protein stops being itself.

# ------------------------------------------------------------------ resolve
# Standard library only - this cell runs before anything is installed.
import random

LIBRARY = {
    "Trp-cage (20 aa) - the smallest thing that still folds":
        "NLYIQWLKDGGPSSGRPPPS",
    "Villin headpiece (36 aa) - a three-helix bundle":
        "MLSDEDFKAVFGMTRSAFANLPLWKQQNLKKEKGLF",
    "Protein G B1 domain (56 aa) - half sheet, half helix":
        "MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "Ubiquitin (76 aa) - the classic signalling protein":
        "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG",
    "GFP (238 aa) - a beta-barrel, slow on CPU":
        "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQ"
        "HDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNG"
        "IKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK",
}
AA = set("ACDEFGHIKLMNPQRSTVWY")

if protein.startswith("paste"):
    NATIVE = "".join(my_own_sequence.split()).upper()
    LABEL  = f"your own sequence ({len(NATIVE)} aa)"
    if not NATIVE:
        raise SystemExit("\nYou chose 'paste my own below' but left the box empty.\n"
                         "Either paste a sequence into my_own_sequence, or pick one\n"
                         "from the protein list.")
    bad = sorted(set(NATIVE) - AA)
    if bad:
        raise SystemExit("\nThat sequence contains letters that are not standard amino\n"
                         "acids: " + " ".join(bad) + "\nUse one-letter code "
                         "(ACDEFGHIKLMNPQRSTVWY), no gaps, no numbers.")
    if len(NATIVE) < 12:
        raise SystemExit(f"\nThat sequence is only {len(NATIVE)} residues. Use at least 12.")
else:
    NATIVE = LIBRARY[protein]
    LABEL  = protein.split(" -")[0]

SWEEP_SEQ   = LIBRARY[sweep_protein]
SWEEP_LABEL = sweep_protein.split(" -")[0]

random.seed(seed)
N_MASK   = max(1, int(len(NATIVE) * mask_fraction))
MASK_IDX = sorted(random.sample(range(len(NATIVE)), N_MASK))
MASKED   = "".join("_" if i in set(MASK_IDX) else a for i, a in enumerate(NATIVE))

print("=" * 64)
print(f"  protein       {LABEL}   ({len(NATIVE)} residues)")
print(f"  deleting      {N_MASK} residues ({N_MASK / len(NATIVE):.0%})")
print(f"  temperature   {temperature}      seed {seed}")
print(f"  sweep         {('yes, on ' + SWEEP_LABEL) if run_sweep else 'no'}")
print("=" * 64)
print("\nwhat the model will see:")
for i in range(0, len(MASKED), 60):
    print("   " + MASKED[i:i + 60])

if len(NATIVE) > 150:
    print("\nNOTE: that is a long protein. Fine on a GPU, slow on CPU.")

print("\nNow: Runtime -> Run all.")


In [ ]:
#@title Setup - install ESM3 and load the model  (several minutes)
%%time
import subprocess, sys, os

print('installing esm ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'esm', 'py3Dmol', 'matplotlib'], check=True)

import torch
from esm.models.esm3 import ESM3
from esm.sdk.api import ESMProtein, GenerationConfig

# T4 GPUs have no bfloat16; float32 is the safe choice everywhere.
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print('GPU:', torch.cuda.get_device_name(0))
else:
    DEVICE = 'cpu'
    print('No GPU - running on CPU. Use the shorter proteins.')

print('downloading model weights (~5 GB, once) ...')
model = ESM3.from_pretrained('esm3-sm-open-v1').to(DEVICE).float()

# smoke test - fail now rather than three cells later
_p = model.generate(ESMProtein(sequence='_' * 12),
                    GenerationConfig(track='sequence', num_steps=12, temperature=0.7))
assert isinstance(_p.sequence, str) and len(_p.sequence) == 12, 'smoke test failed'
print(f'\nready on {DEVICE}  (test output: {_p.sequence})')


---
## The experiment

The masked positions go in, ESM3 fills them back, and three numbers come out:

| Number | What it tells you |
|---|---|
| **overall identity** | how much of the whole sequence matches the original |
| **identity at masked positions** | whether it *recovered* the original residues or *invented* new ones |
| **top residue %** | sanity check, printed next to the same number for the *original* protein. If your design is far above it, the model has stopped designing and started repeating one residue |

The second row is the one that matters. The first is inflated by the residues
you never deleted.

A note on the third: there is no universal "normal" here. Most proteins sit near
8–12%, but Trp-cage is 25% proline and that is simply what Trp-cage is. So the
comparison is always against the protein you chose, never a fixed number.


In [ ]:
#@title Mask and regenerate
import collections, time
from esm.sdk.api import ESMProtein, GenerationConfig

t0 = time.time()
p = ESMProtein(sequence=MASKED)
p = model.generate(p, GenerationConfig(track='sequence',
                                       num_steps=max(len(MASK_IDX), 1),
                                       temperature=temperature))
p = model.generate(p, GenerationConfig(track='structure', num_steps=8))
elapsed = time.time() - t0

DESIGNED = p.sequence
overall   = sum(a == b for a, b in zip(NATIVE, DESIGNED)) / len(NATIVE)
at_masked = sum(NATIVE[i] == DESIGNED[i] for i in MASK_IDX) / max(len(MASK_IDX), 1)
aa, n = collections.Counter(DESIGNED).most_common(1)[0]
top = n / len(DESIGNED)

# Judge composition against THIS protein, not a fixed cutoff. Short proteins are
# naturally lopsided - Trp-cage is 25% proline - so a flat threshold would call
# the real sequence collapsed.
nat_aa, nat_n = collections.Counter(NATIVE).most_common(1)[0]
nat_top = nat_n / len(NATIVE)

p.to_pdb('design.pdb')

print(f'{LABEL}   {len(NATIVE)} residues, {len(MASK_IDX)} masked ({mask_fraction:.0%})')
print(f'took {elapsed:.1f}s on {DEVICE}\n')
print(f'  overall identity             {overall:6.1%}')
print(f'  identity at masked positions {at_masked:6.1%}')
print(f'  most common residue          {aa} at {top:5.1%}   '
      f'(the original is {nat_aa} at {nat_top:.1%})')
if top > max(0.20, nat_top + 0.15):
    print('      ^ far above the original - the model has collapsed into repeats')
elif top > nat_top + 0.08:
    print('      ^ somewhat above the original - starting to drift')
else:
    print('      ^ normal, this still looks like a protein')
print(f'\noriginal : {NATIVE}')
print(f'designed : {DESIGNED}')
print('changed  : ' + ''.join(' ' if a == b else '^' for a, b in zip(NATIVE, DESIGNED)))


---
## Did the fold survive?

ESM3 also predicts a structure for the sequence it wrote. Drag to rotate.
Compare what you see with the real fold — a β-barrel for GFP, a β-grasp for
ubiquitin, a three-helix bundle for villin.


In [ ]:
#@title View the designed structure
import subprocess, sys
try:
    import py3Dmol
except ModuleNotFoundError:          # fresh runtime, or the setup cell not run
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'py3Dmol'], check=True)
    import py3Dmol

view = py3Dmol.view(width=720, height=520)
view.addModel(open('design.pdb').read(), 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.setBackgroundColor('white')
view.zoomTo()
view.show()
print('blue = N-terminus, red = C-terminus')


---
## Find the breaking point

One run tells you little. This sweeps the masking fraction and plots what
happens, so you can see *where* the protein stops being itself.

Watch the two identity lines diverge, and watch the composition line rise — that
rise is the model giving up and padding with one repeated residue.


In [ ]:
#@title Sweep the masking fraction
import random, collections
import matplotlib.pyplot as plt
from esm.sdk.api import ESMProtein, GenerationConfig

if not run_sweep:
    print('run_sweep is off - set it to True in the choices cell to run this.')
else:
    seq = SWEEP_SEQ
    fracs = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8]
    ov, am, tp = [], [], []
    for f in fracs:
        random.seed(seed)
        idx = sorted(random.sample(range(len(seq)), max(1, int(len(seq) * f))))
        m = ''.join('_' if i in set(idx) else a for i, a in enumerate(seq))
        q = model.generate(ESMProtein(sequence=m),
                           GenerationConfig(track='sequence',
                                            num_steps=max(len(idx), 1),
                                            temperature=temperature))
        s = q.sequence
        ov.append(sum(a == b for a, b in zip(seq, s)) / len(seq))
        am.append(sum(seq[i] == s[i] for i in idx) / max(len(idx), 1))
        tp.append(collections.Counter(s).most_common(1)[0][1] / len(s))
        print(f'  {f:>4.0%} masked -> overall {ov[-1]:5.1%}   '
              f'at masked {am[-1]:5.1%}   top residue {tp[-1]:5.1%}')

    nat_top = collections.Counter(seq).most_common(1)[0][1] / len(seq)

    plt.figure(figsize=(7, 4.5))
    x = [f * 100 for f in fracs]
    plt.plot(x, [v * 100 for v in ov], 'o-', label='overall identity')
    plt.plot(x, [v * 100 for v in am], 's-', label='identity at masked positions')
    plt.plot(x, [v * 100 for v in tp], '^--', label='most common residue %')
    plt.axhline(nat_top * 100, color='green', ls=':', lw=1.5)
    plt.text(x[0], nat_top * 100 + 2,
             f'the real protein sits at {nat_top:.0%}', fontsize=8, color='green')
    plt.xlabel('% of sequence masked'); plt.ylabel('%')
    plt.title(SWEEP_LABEL); plt.legend(); plt.grid(alpha=0.3); plt.ylim(0, 100)
    plt.show()


---
## Check it with an independent model

ESM3 folded its own sequence, so of course it agrees with itself. A genuinely
independent predictor is a far stronger test — and
[AlphaFold Server](https://alphafoldserver.com) is free and takes about a minute.


In [ ]:
#@title Sequences for AlphaFold Server
print('Go to  https://alphafoldserver.com   (sign in with a Google account)')
print('Add ONE protein entity and paste the designed sequence into it.\n')
print('=' * 64)
print(f'YOUR DESIGN ({len(DESIGNED)} aa) - this is the one to submit')
print('=' * 64)
print(DESIGNED)
print()
print('=' * 64)
print(f'THE ORIGINAL ({len(NATIVE)} aa) - submit this too if you have time,')
print('as a control: it is the fold your design is supposed to have kept')
print('=' * 64)
print(NATIVE)
print("""
Then Continue, then Submit.

What to look for:
  * is the overall shape the same as the original fold?
  * pLDDT colouring - blue is confident, orange and yellow are not. A design
    that collapsed will be mostly orange, and often a long floppy loop.
  * the PAE plot - dark means AlphaFold knows where two parts sit relative to
    each other. A design that lost its fold goes pale everywhere.

If AlphaFold disagrees with ESM3, that disagreement is the most useful result
you will get today.
""")


In [ ]:
#@title Download your design
from google.colab import files
files.download('design.pdb')


---
## Things to try, and one thing to look for

Go back to the **choices cell at the top**, change one thing, and
**Runtime → Run all** again.

- **Run ubiquitin at 80% masking.** Look at the identity at masked positions.
  Then ask yourself what the model is actually doing. This is the most
  important result in the notebook.
- **Compare ubiquitin with Trp-cage** at the same masking fraction. Why might
  one be recalled far better than the other?
- **Drop the temperature to 0.3.** The output gets *more* confident and *less*
  protein-like. Why would that be?
- **Push to 90% masking.** At some point you are no longer redesigning a
  protein, you are asking the model to invent one. Watch what it does.
- **Change the seed only.** Same protein, same fraction, different residues
  deleted. How much does the answer move?
- **Paste in a protein you care about** and find its breaking point.

## For the discussion

Bring one number: the masking fraction at which *your* protein stopped being
itself, and how you decided that.

---

ESM3: Hayes et al., *Science* 2025. Model `esm3-sm-open-v1`, MIT licence.
